<a name='1'></a>
## Résumer un dialogue sans Prompt Engineering

Dans ce cas d’utilisation, vous allez générer un résumé d’un dialogue avec le modèle de langage pré-entraîné (LLM) Qwen3.5-0.8B de Hugging Face. La liste des modèles disponibles dans le package transformers de Hugging Face se trouve [ici](https://huggingface.co/docs/transformers/index).

Chargeons maintenant quelques dialogues simples issus du dataset [DialogSum](https://huggingface.co/datasets/knkarthick/dialogsum) de Hugging Face. Ce dataset contient plus de 10 000 dialogues accompagnés de leurs résumés manuellement annotés ainsi que de leurs thématiques.


In [1]:
from datasets import load_dataset
from transformers import pipeline


In [2]:
huggingface_dataset_name = "knkarthick/dialogsum"
dataset = load_dataset(huggingface_dataset_name)

In [3]:
example_indices = [40, 200]

dash_line = '-'.join('' for x in range(100))

for i, index in enumerate(example_indices):
    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print('INPUT DIALOGUE:')
    print(dataset['test'][index]['dialogue'])
    print(dash_line)
    print('BASELINE HUMAN SUMMARY:')
    print(dataset['test'][index]['summary'])
    print(dash_line)
    print()

---------------------------------------------------------------------------------------------------
Example  1
---------------------------------------------------------------------------------------------------
INPUT DIALOGUE:
#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.
---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
---------------------------------------------------------------------------------------------------

---------------------------------------------------------------------------------------------------
Exa

In [4]:
# Chargement du modele preentraine Qwen3.5-0.8B (comme dans le notebook RAG).
# On le charge UNE seule fois, puis on reutilise le pipeline `pipe`.
pipe = pipeline("image-text-to-text", model="Qwen/Qwen3.5-0.8B")


def generate(prompt, max_new_tokens=50):
    """Genere du texte avec Qwen3.5-0.8B a partir d'un prompt (str)."""
    messages = [
        {"role": "user", "content": [{"type": "text", "text": prompt}]},
    ]
    history = pipe(text=messages, max_new_tokens=max_new_tokens)
    return history[-1]['generated_text'][-1]['content']


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [5]:
# Sans prompt engineering
for i, index in enumerate(example_indices):
    dialogue = dataset['test'][index]['dialogue']

    output = generate(dialogue, max_new_tokens=50)

    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print(f'INPUT PROMPT:\n{dialogue}')
    print(dash_line)

    summary = dataset['test'][index]['summary']
    print(f'BASELINE HUMAN SUMMARY:\n{summary}')

    print(dash_line)
    print(f'MODEL GENERATION - WITHOUT PROMPT ENGINEERING:\n{output}\n')


Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


---------------------------------------------------------------------------------------------------
Example  1
---------------------------------------------------------------------------------------------------
INPUT PROMPT:
#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.
---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
---------------------------------------------------------------------------------------------------
MODEL GENERATION - WITHOUT PROMPT ENGINEERING:
Based on the conversation provided, here is the breakdown o

In [6]:
# Minimal prompt engineering -> c'est a peine mieux
for i, index in enumerate(example_indices):
    dialogue = dataset['test'][index]['dialogue'] + '\nSummary:'
    summary = dataset['test'][index]['summary']

    output = generate(dialogue, max_new_tokens=50)

    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print(f'INPUT PROMPT:\n{dialogue}')
    print(dash_line)
    print(f'BASELINE HUMAN SUMMARY:\n{summary}')
    print(dash_line)
    print(f'MODEL GENERATION - WITHOUT PROMPT ENGINEERING:\n{output}\n')


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


---------------------------------------------------------------------------------------------------
Example  1
---------------------------------------------------------------------------------------------------
INPUT PROMPT:
#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.
Summary:
---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
---------------------------------------------------------------------------------------------------
MODEL GENERATION - WITHOUT PROMPT ENGINEERING:
Based on the conversation provided, here is the su

In [7]:
# Exercice 1: Faire du prompt engineering afin d'ameliorer la sortie. Pour de meilleures performances, prompter en anglais !
for i, index in enumerate(example_indices):
    dialogue = dataset['test'][index]['dialogue']
    summary = dataset['test'][index]['summary']

    prompt = f"""
Summarize the following conversation.

{dialogue}

Summary:
        """

    output = generate(prompt, max_new_tokens=50)

    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print(f'INPUT PROMPT:\n{prompt}')
    print(dash_line)
    print(f'BASELINE HUMAN SUMMARY:\n{summary}')
    print(dash_line)
    print(f'MODEL GENERATION - ZERO SHOT:\n{output}\n')


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


---------------------------------------------------------------------------------------------------
Example  1
---------------------------------------------------------------------------------------------------
INPUT PROMPT:

Summarize the following conversation.

#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.

Summary:
        
---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
---------------------------------------------------------------------------------------------------
MODEL GENERATION - ZERO SHOT:
Tom is late for a

In [8]:
# Exercice 2: Faire du prompt engineering + One-Shot
def make_prompt(example_indices_full, example_index_to_summarize):
    prompt = ''
    for index in example_indices_full:
        dialogue = dataset['test'][index]['dialogue']
        summary = dataset['test'][index]['summary']

        prompt += f"""
Dialogue:

{dialogue}

What was going on?
{summary}


"""

    dialogue = dataset['test'][example_index_to_summarize]['dialogue']

    prompt += f"""
Dialogue:

{dialogue}

What was going on?
"""

    return prompt


example_indices_full = [40]
example_index_to_summarize = 200

one_shot_prompt = make_prompt(example_indices_full, example_index_to_summarize)

print(one_shot_prompt)



Dialogue:

#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.

What was going on?
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.



Dialogue:

#Person1#: Have you considered upgrading your system?
#Person2#: Yes, but I'm not sure what exactly I would need.
#Person1#: You could consider adding a painting program to your software. It would allow you to make up your own flyers and banners for advertising.
#Person2#: That would be a definite bonus.
#Person1#: You might also want to upgrade your hardware because it is pretty outdated now.
#Person2#: How can we do that?
#Person1#: You'd probably need a faster processor, to begin with. And you also ne

In [9]:
summary = dataset['test'][example_index_to_summarize]['summary']

output = generate(one_shot_prompt, max_new_tokens=50)

print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'MODEL GENERATION - ONE SHOT:\n{output}')


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.

---------------------------------------------------------------------------------------------------
MODEL GENERATION - ONE SHOT:
#Person1# is suggesting that #Person2# upgrade their computer system to improve their software capabilities.

#Person2# is hesitant to accept the upgrade because they are unsure what specific changes they would need.

#Person1# offers a


In [10]:
# Exercice 3: Faire du prompt engineering + Few-Shot. A partir de combien d'exemples il est inutile d'en ajouter ?

example_indices_full = [40, 80, 120]
example_index_to_summarize = 200

few_shot_prompt = make_prompt(example_indices_full, example_index_to_summarize)

print(few_shot_prompt)


Dialogue:

#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.

What was going on?
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.



Dialogue:

#Person1#: May, do you mind helping me prepare for the picnic?
#Person2#: Sure. Have you checked the weather report?
#Person1#: Yes. It says it will be sunny all day. No sign of rain at all. This is your father's favorite sausage. Sandwiches for you and Daniel.
#Person2#: No, thanks Mom. I'd like some toast and chicken wings.
#Person1#: Okay. Please take some fruit salad and crackers for me.
#Person2#: Done. Oh, don't forget to take napkins disposable plates, cups and picnic blanket.
#Person1#: All set. 

In [11]:
summary = dataset['test'][example_index_to_summarize]['summary']

output = generate(few_shot_prompt, max_new_tokens=50)

print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'MODEL GENERATION - FEW SHOT:\n{output}')


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.

---------------------------------------------------------------------------------------------------
MODEL GENERATION - FEW SHOT:
#Person1# is offering a series of upgrades to #Person2#'s computer system, starting with software (painting programs) and hardware (CD-ROM drives and faster processors).

#Person2# declines the upgrade, stating that they are


# Exercice : Classification de sentiments par prompt engineering

Jusqu'ici on a *résumé*. On va maintenant *classifier* : prédire le sentiment
(`positive` / `negative`) d'un avis, **sans entraînement**, uniquement par le prompt.
On compare deux régimes d'*in-context learning* :

- **Zero-shot** : on décrit la tâche, aucun exemple.
- **Few-shot** : on fournit quelques exemples résolus dans le prompt.

On réutilise la fonction `generate` (Qwen3.5-0.8B) déjà définie plus haut. Comme indiqué,
le modèle répond mieux **en anglais**.


In [12]:
# Petit jeu de test etiquete (equilibre). Verite terrain = `label`.
reviews = [
    ("I absolutely loved this movie, it was brilliant.", "positive"),
    ("A complete waste of time, terrible acting.",        "negative"),
    ("Best purchase I have made this year, highly recommend.", "positive"),
    ("The product broke after two days, very disappointing.",  "negative"),
    ("Fantastic experience, I would come back again.",    "positive"),
    ("Awful service and the food was cold.",              "negative"),
    ("This book changed my life, truly inspiring.",       "positive"),
    ("Boring, predictable and far too long.",             "negative"),
]


def accuracy(predict_fn):
    ok = 0
    for text, label in reviews:
        pred = predict_fn(text).strip().lower()
        norm = "positive" if "pos" in pred else ("negative" if "neg" in pred else pred)
        ok += (norm == label)
        print(f"{label:8} | pred={pred:10} | {text[:45]}")
    print(f"\nAccuracy : {ok}/{len(reviews)} = {100*ok/len(reviews):.0f}%")
    return ok / len(reviews)


## Zero-shot

**À compléter :** rédigez un prompt *zero-shot* qui demande au modèle de classer l'avis
en `positive` ou `negative`.

In [13]:
def predict_zero_shot(text):
    prompt = (
        "Classify the sentiment of the following review as positive or negative.\n"
        f"Review: {text}\n"
        "Sentiment:"
    )
    return generate(prompt)

print("=== ZERO-SHOT ===")
acc_zero = accuracy(predict_zero_shot)


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== ZERO-SHOT ===


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


positive | pred=positive   | I absolutely loved this movie, it was brillia


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

positive | pred=the sentiment of the review is **positive**.

the phrase "fantastic experience" directly indicates a high level of satisfaction, and the concluding statement "i would come back again" expresses strong intention to return to the service, both of which are clear indicators | Fantastic experience, I would come back again


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


negative | pred=the sentiment of the review is **negative**.

the review explicitly states that the service was "awful" and the food was "cold," which are clear indicators of dissatisfaction. | Awful service and the food was cold.


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


positive | pred=the sentiment of the review is **positive**.

the review uses words like "changed my life," "truly inspiring," and expresses a strong emotional response, which are all indicators of a positive sentiment. | This book changed my life, truly inspiring.
negative | pred=the sentiment of the review is **negative**.

the words "boring," "predictable," and "far too long" all convey dissatisfaction with the content or experience, indicating a negative opinion. | Boring, predictable and far too long.

Accuracy : 8/8 = 100%


## Few-shot

**À compléter :** ajoutez 2 à 4 exemples résolus *avant* l'avis à classer (few-shot).
Observez l'effet sur l'accuracy et sur la *stabilité* du format de sortie.

In [14]:
def predict_few_shot(text):
    prompt = (
        "Classify the sentiment of each review as positive or negative.\n"
        "Review: The plot was amazing and the cast was perfect.\nSentiment: positive\n"
        "Review: I regret buying this, it stopped working immediately.\nSentiment: negative\n"
        "Review: What a delightful and heartwarming story.\nSentiment: positive\n"
        f"Review: {text}\nSentiment:"
    )
    return generate(prompt)

print("=== FEW-SHOT ===")
acc_few = accuracy(predict_few_shot)
print(f"\nGain few-shot vs zero-shot : {100*(acc_few-acc_zero):+.0f} points")


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== FEW-SHOT ===


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


positive | pred=sentiment: positive | I absolutely loved this movie, it was brillia


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

positive | pred=sentiment: positive | Fantastic experience, I would come back again


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


negative | pred=review: awful service and the food was cold.
sentiment: negative | Awful service and the food was cold.


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


positive | pred=sentiment: **positive** | This book changed my life, truly inspiring.
negative | pred=review: boring, predictable and far too long.
sentiment: negative | Boring, predictable and far too long.

Accuracy : 8/8 = 100%

Gain few-shot vs zero-shot : +0 points
